# Développement de l'API REST avec FastAPI

## Objectif

Après avoir entraîné, évalué et sélectionné le meilleur modèle, celui-ci doit être rendu accessible afin de pouvoir être utilisé par des applications externes.

Dans un environnement de production, les prédictions ne sont pas réalisées directement depuis un notebook Jupyter. Elles sont effectuées via une API REST capable de recevoir de nouvelles données, de charger automatiquement le meilleur modèle sauvegardé et de retourner une prédiction.

Cette étape marque le début de la phase MLOps du projet. L'API développée avec FastAPI constituera la base du déploiement avec Docker, Kubernetes et de l'intégration continue (CI/CD).

# Architecture de l'API REST

## Objectif

Avant de développer l'API REST, il est nécessaire de mettre en place une architecture claire et modulaire. Cette organisation permet de séparer les différentes responsabilités de l'application, ce qui facilite son développement, sa maintenance et son évolution.

L'API est organisée autour de plusieurs fichiers, chacun ayant un rôle bien défini.

### Description des fichiers

- **`app/`** : dossier principal contenant l'ensemble du code source de l'API REST.

- **`__init__.py`** : indique que le dossier `app` est un package Python. Il permet à Python de reconnaître ce dossier comme un module pouvant être importé dans les autres fichiers du projet.

- **`main.py`** : point d'entrée de l'application FastAPI. Il initialise l'API, définit les différentes routes (endpoints) et lance le serveur permettant aux utilisateurs d'envoyer des requêtes de prédiction.

- **`model_loader.py`** : responsable du chargement automatique du meilleur modèle sauvegardé (`best_model.pkl` ou `best_model.keras`). Il évite de recharger le modèle à chaque requête, ce qui améliore les performances de l'API.

- **`prediction.py`** : contient toute la logique de prédiction. Il reçoit les données envoyées par l'utilisateur, applique le modèle chargé et retourne le résultat de la prédiction.

- **`schemas.py`** : définit les schémas de données d'entrée et de sortie à l'aide de la bibliothèque Pydantic. Il permet de valider automatiquement les données reçues avant qu'elles ne soient utilisées par le modèle.



Cette organisation respecte les bonnes pratiques de développement logiciel et de MLOps. La séparation des responsabilités rend l'application plus lisible, plus facilement testable et simplifie les étapes de déploiement avec Docker, Kubernetes et les outils d'intégration continue.

In [1]:
import os

# Création des dossiers
os.makedirs("app", exist_ok=True)
os.makedirs("best_model", exist_ok=True)

# Création des fichiers de l'API
files = [
    "app/__init__.py",
    "app/main.py",
    "app/model_loader.py",
    "app/prediction.py",
    "app/schemas.py"
]

for file in files:
    if not os.path.exists(file):
        open(file, "w").close()

print("Structure de l'API créée avec succès.")

Structure de l'API créée avec succès.


# Installation de FastAPI

## Objectif

FastAPI est un framework Python moderne permettant de développer rapidement des API REST performantes.

Il sera utilisé pour charger automatiquement le meilleur modèle de détection d'intrusions et exposer un service de prédiction accessible par des applications externes.

In [2]:
pip install fastapi uvicorn


[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Création du point d'entrée de l'API

## Objectif

Le fichier `main.py` constitue le point d'entrée principal de l'application FastAPI.

Dans un premier temps, une route d'accueil est créée afin de vérifier que l'API démarre correctement avant d'intégrer le modèle de Machine Learning.

# Développement du fichier `main.py`

## Objectif

Le fichier `main.py` constitue le point d'entrée principal de l'API REST. Il est responsable de l'initialisation de l'application FastAPI et de la définition des différents endpoints qui permettront aux utilisateurs d'interagir avec le modèle de détection d'intrusions.

Dans un premier temps, une route d'accueil est mise en place afin de vérifier que l'API démarre correctement et qu'elle est accessible depuis un navigateur ou un client HTTP. Cette étape permet de valider le bon fonctionnement de l'environnement FastAPI avant d'intégrer le modèle de Machine Learning et les fonctionnalités de prédiction.

Le fichier `main.py` sera progressivement enrichi au cours des prochaines étapes pour intégrer le chargement automatique du modèle, la réception des données d'entrée et le retour des prédictions.

### Explication du code

Le fichier `main.py` commence par importer la classe `FastAPI`, qui permet de créer une application web capable de recevoir et de traiter des requêtes HTTP.

Une instance de l'application est ensuite créée à l'aide de la classe `FastAPI`. Les informations telles que le nom, la description et la version de l'API sont renseignées afin de générer automatiquement une documentation interactive.

La route d'accueil (`GET /`) est ensuite définie. Lorsqu'un utilisateur accède à cette route, l'API retourne un message confirmant qu'elle est correctement démarrée et prête à recevoir des requêtes. Cette première route constitue un test simple permettant de vérifier le bon fonctionnement de l'application avant d'ajouter des fonctionnalités plus avancées.

### Lancement du serveur avec Uvicorn

#### Objectif

Après le développement du fichier `main.py`, il est nécessaire de lancer l'application afin de vérifier son bon fonctionnement.

Pour cela, FastAPI utilise **Uvicorn**, un serveur ASGI (Asynchronous Server Gateway Interface) capable d'exécuter des applications web Python de manière performante.

Uvicorn joue le rôle d'intermédiaire entre les clients (navigateur web, application mobile, logiciel, etc.) et l'application FastAPI. Il reçoit les requêtes HTTP, les transmet à l'API puis renvoie les réponses aux utilisateurs.

Cette étape permet de démarrer localement l'API et de tester son fonctionnement avant son déploiement dans un environnement de production.

# Développement du fichier `model_loader.py`

## Objectif

Une fois l'API créée, il est nécessaire de lui permettre d'utiliser le modèle de Machine Learning sélectionné.

Le rôle du fichier `model_loader.py` est de charger automatiquement le meilleur modèle sauvegardé lors du démarrage de l'application. Ainsi, le modèle est chargé une seule fois en mémoire, puis réutilisé pour toutes les requêtes de prédiction.

Cette approche améliore considérablement les performances de l'API en évitant de recharger le modèle à chaque nouvelle requête.

Par ailleurs, la séparation du chargement du modèle dans un fichier dédié respecte les bonnes pratiques de développement logiciel en isolant cette responsabilité du reste de l'application.

# Développement du fichier `schemas.py`

## Objectif

Le fichier `schemas.py` est chargé de définir la structure des données échangées entre le client et l'API.

Il utilise la bibliothèque **Pydantic**, intégrée à FastAPI, afin de décrire le format des données d'entrée attendues par le modèle de Machine Learning ainsi que le format des réponses retournées par l'API.

La validation automatique des données constitue une étape essentielle pour garantir la robustesse de l'application. Elle permet de détecter les erreurs de saisie avant l'exécution du modèle et d'assurer que les prédictions sont réalisées uniquement sur des données conformes.

En outre, les schémas définis dans ce fichier sont utilisés par FastAPI pour générer automatiquement une documentation interactive, facilitant ainsi l'utilisation de l'API.

# Développement du fichier `prediction.py`

## Objectif

Le fichier `prediction.py` contient la logique de prédiction de l'API.

Après validation des données par `schemas.py`, ce fichier est chargé de préparer les données dans le format attendu par le modèle de Machine Learning, d'exécuter la prédiction et de retourner un résultat compréhensible.

Cette séparation entre les routes de l'API (`main.py`) et la logique de prédiction (`prediction.py`) respecte les bonnes pratiques de développement logiciel. Elle améliore la lisibilité du projet, facilite les tests et simplifie les évolutions futures.

## Explication du code

Le fichier commence par importer NumPy, utilisé pour convertir les données reçues en tableau numérique compatible avec le modèle de Machine Learning.

Un dictionnaire `LABELS` est ensuite défini afin d'établir la correspondance entre les identifiants numériques prédits par le modèle et les noms réels des classes du jeu de données CICIDS2017.

La fonction `predict()` constitue la partie centrale du fichier.

Elle reçoit deux paramètres :

- le modèle de Machine Learning déjà chargé en mémoire ;
- les données validées par le schéma `PredictionInput`.

Les données sont ensuite converties en tableau NumPy de dimension `(1, 47)`, correspondant au format attendu par le modèle Random Forest.

La méthode `predict()` du modèle est alors exécutée afin d'obtenir la classe prédite.

Enfin, le numéro de classe est converti en son libellé grâce au dictionnaire `LABELS`, puis les deux informations sont retournées sous forme de réponse JSON.

# Développement de l'endpoint `/predict`

## Objectif

L'endpoint `/predict` constitue le point d'entrée principal de l'API.

Il permet à un client d'envoyer les caractéristiques d'un flux réseau afin d'obtenir une prédiction réalisée par le modèle de Machine Learning.

Lorsqu'une requête est reçue, FastAPI valide automatiquement les données grâce au schéma `PredictionInput`. Les données validées sont ensuite transmises au fichier `prediction.py`, qui exécute le modèle et retourne la classe prédite ainsi que son libellé.

Cette architecture permet de séparer la gestion des requêtes HTTP de la logique métier, tout en garantissant la fiabilité des prédictions.